# RAG with OpenGenAI Stack - Streaming Responses

This notebook demonstrates **RAG queries using OpenGenAI Stack (OGX)** with the
**Responses API** and **streaming**.

**Prerequisites:**
- Run [rag_pipeline_build.ipynb](../build_rag_pipeline/rag_pipeline_build.ipynb) first to deploy the pipeline and ingest data
- OpenGenAI Stack deployed via [Deploying OGX](https://docs.redhat.com/en/documentation/red_hat_openshift_ai_self-managed/3.3/html-single/working_with_llama_stack/working_with_llama_stack#deploying-llama-stack-server_rag)
- This notebook must run **inside the cluster** (RHOAI notebook server)

## What This Notebook Covers

1. **OGX Setup** — Connect to OGX using the OGX client
2. **Verify Services** — Check OGX endpoint and vector store
3. **RAG with Streaming** — Responses API with `file_search` for automatic RAG
4. **Side-by-Side Comparison** — RAG vs. Direct LLM
5. **Interactive Streaming Q&A** — Live query loop with streaming
6. **Inspect Retrieved Chunks** — View what the vector store returns

In [ ]:
# Install required packages
!pip install -q "ogx-client>=1.0.0"

---
## 1. Configuration

These values must match your deployment.

In [ ]:
NAMESPACE = "ray-docling"

# OpenGenAI Stack (OGX)
OGX_SERVICE = "ogx-custom-distribution"
OGX_URL = f"http://{OGX_SERVICE}.{NAMESPACE}.svc.cluster.local:8321"
MODEL_NAME = "vllm-inference/mistral-7b-instruct-v0-3"  # From ogx-stack.yaml

# Vector store name (must match the pipeline ingestion)
VECTOR_STORE_NAME = "open_rag_pdfs"

print(f"OGX URL:      {OGX_URL}")
print(f"Model:        {MODEL_NAME}")
print(f"Vector Store: {VECTOR_STORE_NAME}")

---
## 2. Connect to OGX

In [ ]:
from ogx_client import OgxClient

client = OgxClient(base_url=OGX_URL)

print(f"Connected to: {OGX_URL}")

### 2.1 Verify OGX Endpoint

In [ ]:
try:
    models = client.models.list()
    print("\u2705 OGX is running")
    print("Available models:")
    for m in models.data:
        print(f"  - {m.id}")
except Exception as e:
    print(f"\u274c Connection failed: {e}")
    print("Ensure OGX is deployed: oc apply -f ogx-stack.yaml")

### 2.2 Find Vector Store

In [ ]:
vector_store_id = None

stores = client.vector_stores.list()
for store in stores.data:
    if store.name == VECTOR_STORE_NAME:
        vector_store_id = store.id
        print(f"\u2705 Found vector store: {store.name}")
        print(f"   ID:     {store.id}")
        print(f"   Status: {store.status}")
        print(f"   Files:  {store.file_counts.total}")
        break

if vector_store_id is None:
    print(f"\u274c Vector store '{VECTOR_STORE_NAME}' not found.")
    print("Available stores:")
    for store in stores.data:
        print(f"  - {store.name} ({store.id})")
    print("\nRun the ingestion pipeline first.")

---
## 3. RAG with Streaming Responses

The OGX **Responses API** with `file_search` handles the full RAG pipeline in one
call: it searches the vector store, retrieves relevant chunks, and generates a
response — all with streaming.

In [ ]:
def rag_query_streaming(
    question, max_tokens=512, temperature=0.1, include_sources=True
):
    """RAG with streaming using the OGX Responses API and file_search."""
    print(f"\U0001f50d Query: {question}\n")
    print("-" * 60)

    full_response = ""
    include = ["file_search_call.results"] if include_sources else []

    stream = client.responses.create(
        model=MODEL_NAME,
        input=question,
        tools=[{"type": "file_search", "vector_store_ids": [vector_store_id]}],
        max_output_tokens=max_tokens,
        temperature=temperature,
        include=include,
        stream=True,
    )

    file_search_results = []

    for event in stream:
        if event.type == "response.output_text.delta":
            print(event.delta, end="", flush=True)
            full_response += event.delta
        elif event.type == "response.completed":
            for output_item in event.response.output:
                if getattr(output_item, "type", None) == "file_search_call":
                    file_search_results = getattr(output_item, "results", None) or []
                    break

    print("\n" + "-" * 60)

    if file_search_results:
        print(f"\n\U0001f4da Sources ({len(file_search_results)}):")
        for i, result in enumerate(file_search_results, 1):
            score = getattr(result, "score", 0)
            filename = getattr(result, "filename", "unknown")
            print(f"  {i}. {filename} (score: {score:.3f})")

    return {
        "question": question,
        "answer": full_response,
        "sources": file_search_results,
    }


print("✅ Streaming RAG function ready")

### 3.1 Test Streaming RAG Query

In [ ]:
question = "What are the main topics covered in the documents?"

result = rag_query_streaming(question)

### 3.2 Another Streaming Query

In [ ]:
result = rag_query_streaming("Summarize the key findings from the documents.")

---
## 4. Side-by-Side Comparison

Compare **RAG with file_search** vs. **Direct LLM** (no retrieval)

In [ ]:
question = "What are the main topics covered in the documents?"

print("=" * 80)
print("RAG WITH FILE_SEARCH (Streaming)")
print("=" * 80)
rag_result = rag_query_streaming(question)

print("\n" + "=" * 80)
print("DIRECT LLM WITHOUT CONTEXT (Streaming)")
print("=" * 80)
print("-" * 60)

direct_stream = client.responses.create(
    model=MODEL_NAME,
    input=question,
    max_output_tokens=512,
    temperature=0.1,
    stream=True,
)

for event in direct_stream:
    if event.type == "response.output_text.delta":
        print(event.delta, end="", flush=True)

print("\n" + "-" * 60)

print("\n" + "=" * 80)
print("OBSERVATION")
print("=" * 80)
print("The RAG answer is grounded in actual document content,")
print("while the direct answer is generic and speculative.")

---
## 5. Interactive Streaming Q&A

Ask questions and see answers stream in real-time. Type `quit` to exit.

In [ ]:
print("\n\U0001f680 Interactive RAG Query Session (Streaming Enabled)")
print("Type your questions and watch answers stream in real-time!")
print("Type 'quit', 'exit', or 'q' to stop.\n")

while True:
    question = input("\n\u2753 Your question: ").strip()
    if question.lower() in ("quit", "exit", "q"):
        print("\n\U0001f44b Goodbye!")
        break
    if not question:
        continue

    try:
        rag_query_streaming(question)
    except Exception as e:
        print(f"\n\u274c Error: {e}")
        print("Please try again.")

---
## 6. Inspect Retrieved Chunks

Use the vector store search API directly to see what chunks are retrieved.

In [ ]:
question = "What are the main topics?"

results = client.vector_stores.search(
    vector_store_id=vector_store_id,
    query=question,
    max_num_results=5,
)

print(f"Top {len(results.data)} chunks for: '{question}'\n")
print("=" * 80)

for i, result in enumerate(results.data, 1):
    score = getattr(result, "score", 0)
    filename = getattr(result, "filename", "unknown")
    print(f"\n\U0001f4c4 Chunk {i} (score: {score:.3f})")
    print(f"Source: {filename}")
    print("-" * 80)
    for content in result.content:
        text = getattr(content, "text", "")
        print(text[:400])
        if len(text) > 400:
            print("[...truncated...]")
    print()

---
## 7. Benefits of Using OpenGenAI Stack with the Responses API

✅ **Automatic RAG** - `file_search` handles retrieval + generation in one call  
✅ **Streaming Events** - Real-time output with structured event types  
✅ **Native OGX Client** - `ogx-client` SDK with full vector store and responses support  
✅ **Conversation Continuation** - Use `previous_response_id` for multi-turn  
✅ **Source Attribution** - Built-in annotations from retrieved chunks  
✅ **Extensibility** - Add `web_search`, MCP tools, or function calling alongside `file_search`  

### Next Steps

1. **Multi-Turn Conversations** — Use `previous_response_id` to chain responses
2. **Add Web Search** — Combine `file_search` with `web_search` for broader context
3. **MCP Integration** — Connect external tools via Model Context Protocol
4. **Hybrid Search** — Configure `search_mode="hybrid"` for keyword + vector retrieval

---
## 8. Cleanup

Uncomment to clean up resources.

In [ ]:
# Delete vector store (removes all files, chunks, and embeddings)
# client.vector_stores.delete(vector_store_id=vector_store_id)
# print(f"Deleted vector store: {vector_store_id}")